# Teste: kDN em lotes vs PyHard — mesmo resultado?

Este notebook compara o **kDN** calculado de duas formas:
1. **PyHard** (matriz inteira N×N, de uma vez)
2. **Em lotes** (matriz lote×N por vez)

Usamos uma **amostra pequena** (ex.: 2500 linhas) para o PyHard caber na memória. Calculamos kDN para as **primeiras 100 linhas** pelos dois jeitos e comparamos: os valores devem ser **iguais** (ou quase, por arredondamento).

**Por que deveria dar igual?** Para cada linha, o kDN usa só a lista de distâncias daquela linha até todas as outras. No PyHard essa lista é a linha da matriz N×N. Em lotes, calculamos exatamente essa mesma lista (linha × todas), só que em um pedaço 100×N. A conta é a mesma. Ver `docs/explicacao_simples_lotes.md`.

## 1. Imports e preparar amostra

In [ ]:
import sys
import types
from pathlib import Path

try:
    import pkg_resources
except ModuleNotFoundError:
    _fake = types.ModuleType("pkg_resources")
    _fake.require = lambda pkg: [type("D", (), {"version": "0.0.0"})()]
    sys.modules["pkg_resources"] = _fake

import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances

BASE = Path("../").resolve()
DATASETS = BASE / "data" / "datasets"
pyhard_path = (BASE / "pyhard").resolve()
if str(pyhard_path) not in sys.path:
    sys.path.insert(0, str(pyhard_path))
for m in list(sys.modules.keys()):
    if m.startswith("pyhard"):
        del sys.modules[m]

from pyhard.measures import ClassificationMeasures

# Amostra pequena para o PyHard rodar (matriz N×N com N=2500)
SAMPLE_SIZE = 2500
K = 10
N_COMPARE = 100  # comparar as primeiras 100 linhas

conjunto = pd.read_csv(DATASETS / "diabetes_conjunto.csv", nrows=SAMPLE_SIZE + 500)
original = pd.read_csv(DATASETS / "diabetes_original.csv")
id_to_y = original.set_index("encounter_id")["y"]
conjunto["y"] = conjunto["encounter_id"].map(id_to_y)
conjunto = conjunto.dropna(subset=["y"]).head(SAMPLE_SIZE).copy()
conjunto["y"] = conjunto["y"].astype(int)

EXCLUDE = {"encounter_id", "patient_nbr", "origem", "problema_qualidade"}
feature_cols = [c for c in conjunto.columns if c not in EXCLUDE and c != "y"]
df_work = conjunto[feature_cols + ["y"]].copy()
for col in df_work.columns:
    if col == "y":
        continue
    if df_work[col].dtype.name == "category":
        df_work[col] = df_work[col].cat.codes
    elif not pd.api.types.is_numeric_dtype(df_work[col]):
        df_work[col] = pd.Categorical(df_work[col].astype(str)).codes
valid = df_work.notna().all(axis=1)
df_pyhard = df_work.loc[valid].reset_index(drop=True)
df_pyhard["y"] = df_pyhard["y"].astype(int)

N = len(df_pyhard)
X = df_pyhard[feature_cols].to_numpy().astype(np.float64)
y = df_pyhard["y"].values.astype(int)

print(f"Amostra: {N} linhas, {X.shape[1]} features")
print(f"Vamos comparar kDN nas primeiras {N_COMPARE} linhas (k={K})")

## 2. kDN com PyHard (matriz inteira)

In [ ]:
# PyHard constrói a matriz N×N internamente e calcula kDN para todas as linhas
measures = ClassificationMeasures(df_pyhard, target_col="y")
kdn_pyhard_all = measures.k_disagreeing_neighbors(k=K, distance="gower")
kdn_pyhard = kdn_pyhard_all[:N_COMPARE]
print("PyHard (primeiras 100):", kdn_pyhard[:5].round(4), "...")

## 3. kDN em lotes (só para as primeiras 100 linhas)

Calculamos distâncias **apenas** das primeiras 100 linhas para **todas** as N. Matriz 100×N. A partir disso, a mesma regra: ordenar, pegar k vizinhos, proporção de classes diferentes.

In [ ]:
def gower_query_ref(X_query, X_ref):
    """Distância de Gower: (nq, nr). Só numérico."""
    nq, n_feat = X_query.shape
    nr = X_ref.shape[0]
    out = np.zeros((nq, nr), dtype=np.float64)
    for j in range(n_feat):
        rng = max(np.ptp(X_ref[:, j]), 1e-8)
        out += pairwise_distances(X_query[:, j:j+1], X_ref[:, j:j+1], metric="manhattan") / rng
    return out / n_feat

# Query = primeiras N_COMPARE linhas; ref = todas as N
X_query = X[:N_COMPARE]
y_query = y[:N_COMPARE]
dist_batch = gower_query_ref(X_query, X)
# dist_batch[i, j] = distância da linha i (query) para a linha j (ref)
# Para a linha i, dist_batch[i, :] é exatamente o que seria a linha i da matriz N×N

idx_sorted = np.argsort(dist_batch, axis=1)
# PyHard usa k+1 (self + k vizinhos); v[0]=self, v[1:] = k vizinhos; kDN = mean(v[1:] != v[0])
kdn_batched = np.zeros(N_COMPARE)
for i in range(N_COMPARE):
    ind = idx_sorted[i, : K + 1]  # self (dist 0) + k vizinhos
    yi = y_query[i]
    neigh_labels = y[ind]
    kdn_batched[i] = np.sum(neigh_labels[1:] != yi) / K

print("Em lotes (primeiras 100):", kdn_batched[:5].round(4), "...")

## 4. Comparação

In [ ]:
diff = np.abs(kdn_pyhard - kdn_batched)
max_diff = np.max(diff)
mean_diff = np.mean(diff)

print("Diferença absoluta (PyHard vs lotes):")
print(f"  Máxima: {max_diff:.2e}")
print(f"  Média:  {mean_diff:.2e}")
print()
if max_diff < 1e-10:
    print("Resultado: os dois métodos dão os MESMOS valores (diferença numérica desprezível).")
else:
    print("Atenção: há diferença. Pode ser implementação (ex.: Gower ligeiramente diferente).")

# Tabela rápida
comparison = pd.DataFrame({
    "linha": np.arange(N_COMPARE),
    "kDN_PyHard": kdn_pyhard,
    "kDN_lotes": kdn_batched,
    "diff": diff,
})
print(comparison.head(10).to_string())